# 02 — Sérsic surface-brightness fit

Forward-model a Sérsic galaxy + Gaussian PSF + noise; recover the parameters by gradient descent (Adam + L-BFGS) and quantify the posterior with NUTS.

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
# Device-agnostic: prefer MPS (Apple GPU) → CUDA → CPU.
# Pass device="cpu" if you need to force the CPU path (e.g. for
# operators that have no MPS kernel yet, or for reproducibility).
device, dtype = gl.config.setup(seed=42)
print(f"using device: {device}")


In [ ]:
npix, dx = 128, 0.05
xy = gl.data.coordinate_grid(npix=npix, deltapix=dx)

true = dict(Ie=5.0, Re=1.0, n=4.0, x0=0.1, y0=-0.2, e1=0.20, e2=-0.10)
galaxy = gl.light.Sersic(**true)

sigma_n = 0.05
clean, image = gl.data.simulate_image(
    galaxy, xy,
    psf_fwhm=0.10, deltapix=dx, psf_size=21,
    noise_sigma=sigma_n, seed=0,
)

ext = (-npix*dx/2, npix*dx/2, -npix*dx/2, npix*dx/2)
gl.viz.side_by_side([clean, image, image - clean],
                    titles=['clean', 'PSF + noise', 'residual'],
                    log=False, extent=ext); plt.show()


## Fit

In [ ]:
class SersicPSF(nn.Module):
    def __init__(self):
        super().__init__()
        self.g = gl.light.Sersic(Ie=2.0, Re=1.5, n=2.5, x0=0., y0=0., e1=0., e2=0.)
    def forward(self, xy):
        k = gl.light.gaussian_psf_kernel(0.10, dx, size=21)
        return gl.light.convolve_psf(self.g(xy), k)

model = SersicPSF()
loss = gl.inference.ReducedChiSquared(sigma=sigma_n, n_params=7)
res = gl.inference.fit(
    model, xy, image, loss,
    lr=0.05, epochs=2500,
    scheduler=gl.inference.optimize.reduce_lr_on_plateau(patience=200, factor=0.7),
    grad_clip=10.0, lbfgs_polish=True,
)
print(f"final chi2/dof = {res.best_loss:.3f}  in {res.duration_s:.1f}s")
for k, v in true.items():
    fv = float(getattr(model.g, k))
    print(f"  {k}: true={v:+.3f}  fit={fv:+.3f}")


In [ ]:
with torch.no_grad():
    fit_img = model(xy)
gl.viz.side_by_side([image, fit_img, image - fit_img],
                    titles=['data', 'best fit', 'residual'],
                    log=False, extent=ext); plt.show()
gl.viz.plot_loss_history(res.loss_history); plt.show()


## Statistical validation

We rely on three complementary checks:

1. **χ²/dof** — should be ≈ 1 if the noise model and the
   functional form are both correct.
2. **Residual distribution** — standardized residuals should
   look ~ N(0,1); a Q-Q plot reveals tail mismatch and the
   Anderson-Darling test gives a p-value-like verdict.
3. **Radial residual profile** — annulus-binned residuals
   exposes systematic radial trends (over- or under-fitting
   the core / outskirts) which the global χ² may miss.

We additionally compute the **AIC** and **BIC** so the user
can compare alternative model choices on the same data.

In [ ]:
data_np   = image.numpy()
model_np  = fit_img.numpy()
chi2_dof  = gl.stats.chi2_per_dof(data_np, model_np, sigma_n, n_params=7)
nll       = gl.stats.gaussian_neg_loglike(data_np, model_np, sigma_n)
aic_val   = gl.stats.aic(nll, n_params=7)
bic_val   = gl.stats.bic(nll, n_params=7, n_samples=data_np.size)
std_res   = gl.stats.standardized_residuals(data_np, model_np, sigma_n)
A2, verdict = gl.stats.anderson_darling_normality(std_res)

print(gl.viz.diagnostics.format_summary({
    'chi2 / dof'             : chi2_dof,
    'Anderson-Darling A^2'   : A2,
    'normality verdict'      : verdict,
    'mean residual / sigma'  : float(std_res.mean()),
    'std  residual / sigma'  : float(std_res.std()),
    'AIC'                    : aic_val,
    'BIC'                    : bic_val,
    'log-likelihood'         : -nll,
    'n_params'               : 7,
    'n_data'                 : int(data_np.size),
}, title='Sersic fit: goodness-of-fit summary'))


In [ ]:
gl.viz.diagnostics.plot_residual_diagnostics(
    data_np, model_np, sigma_n,
    title='Sersic fit residual diagnostics', extent=ext,
)
plt.show()


## NUTS posterior (cached)

In [ ]:
RUN_NUTS = False
csv = Path('cache/posterior_sersic.csv'); csv.parent.mkdir(exist_ok=True)
df = None
if RUN_NUTS:
    import pyro, pyro.distributions as dist
    def pyro_model(xy, image):
        Ie = pyro.sample('Ie', dist.LogNormal(0., 1.))
        Re = pyro.sample('Re', dist.LogNormal(0., 1.))
        n  = pyro.sample('n',  dist.Uniform(0.5, 8.0))
        x0 = pyro.sample('x0', dist.Normal(0., 1.0))
        y0 = pyro.sample('y0', dist.Normal(0., 1.0))
        e1 = pyro.sample('e1', dist.Uniform(-0.7, 0.7))
        e2 = pyro.sample('e2', dist.Uniform(-0.7, 0.7))
        g = gl.light.Sersic(Ie, Re, n, x0, y0, e1, e2)
        pred = g(xy)
        k = gl.light.gaussian_psf_kernel(0.10, dx, size=21)
        pred = gl.light.convolve_psf(pred, k)
        with pyro.plate('data', image.numel()):
            pyro.sample('obs', dist.Normal(pred.flatten(), sigma_n),
                        obs=image.flatten())
    df = gl.inference.run_nuts(pyro_model, xy, image,
                              num_samples=1000, warmup_steps=500,
                              save_path=str(csv))
elif csv.exists():
    import pandas as pd
    df = pd.read_csv(csv)

if df is not None and len(df):
    print(df.describe())
    df_ord = df[['Ie','Re','n','x0','y0','e1','e2']]
    gl.viz.corner_plot(df_ord, truths=[true[k] for k in df_ord.columns]); plt.show()
else:
    print("No cached posterior; set RUN_NUTS=True to generate.")
